# Snake Game

This notebook opens a playable Snake game in a Tkinter window. Use the arrow keys to move the snake and try to eat as many food items as possible without crashing into a wall

In [8]:
from __future__ import annotations

import random
import tkinter as tk
from dataclasses import dataclass


@dataclass
class GameConfig:
    grid_size: int = 20
    cell_size: int = 24
    tick_ms: int = 120


class SnakeGame:
    def __init__(self, master: tk.Tk, config: GameConfig | None = None):
        self.master = master
        self.config = config or GameConfig()
        self.canvas = tk.Canvas(
            master,
            width=self.config.grid_size * self.config.cell_size,
            height=self.config.grid_size * self.config.cell_size,
            bg="#111111",
            highlightthickness=0,
        )
        self.canvas.pack()

        self.status = tk.Label(
            master,
            text="Use arrow keys to move. Press R to restart.",
            anchor="w",
            padx=10,
            pady=6,
        )
        self.status.pack(fill="x")

        self.score_var = tk.StringVar(value="Score: 0")
        self.score_label = tk.Label(master, textvariable=self.score_var, anchor="w", padx=10, pady=6)
        self.score_label.pack(fill="x")

        self.master.bind("<Up>", lambda event: self.set_direction((0, -1)))
        self.master.bind("<Down>", lambda event: self.set_direction((0, 1)))
        self.master.bind("<Left>", lambda event: self.set_direction((-1, 0)))
        self.master.bind("<Right>", lambda event: self.set_direction((1, 0)))
        self.master.bind("r", lambda event: self.reset())
        self.master.bind("R", lambda event: self.reset())

        self.after_id: str | None = None
        self.reset()

    def reset(self) -> None:
        center = self.config.grid_size // 2
        self.snake = [(center, center), (center - 1, center), (center - 2, center)]
        self.direction = (1, 0)
        self.next_direction = (1, 0)
        self.score = 0
        self.game_over = False
        self.place_food()
        self.score_var.set("Score: 0")
        self.status.configure(text="Use arrow keys to move. Press R to restart.")
        self.draw()
        self.schedule_tick()

    def place_food(self) -> None:
        empty_cells = [
            (x, y)
            for x in range(self.config.grid_size)
            for y in range(self.config.grid_size)
            if (x, y) not in self.snake
        ]
        self.food = random.choice(empty_cells)

    def set_direction(self, direction: tuple[int, int]) -> None:
        if self.game_over:
            return
        current_dx, current_dy = self.direction
        new_dx, new_dy = direction
        if (new_dx, new_dy) == (-current_dx, -current_dy):
            return
        self.next_direction = direction

    def schedule_tick(self) -> None:
        if self.after_id is not None:
            self.master.after_cancel(self.after_id)
        self.after_id = self.master.after(self.config.tick_ms, self.tick)

    def tick(self) -> None:
        if self.game_over:
            return

        self.direction = self.next_direction
        head_x, head_y = self.snake[0]
        dx, dy = self.direction
        new_head = (head_x + dx, head_y + dy)

        if self.is_collision(new_head):
            self.game_over = True
            self.status.configure(text=f"Game over. Final score: {self.score}. Press R to restart.")
            self.draw()
            return

        self.snake.insert(0, new_head)
        if new_head == self.food:
            self.score += 1
            self.score_var.set(f"Score: {self.score}")
            self.place_food()
        else:
            self.snake.pop()

        self.draw()
        self.schedule_tick()

    def is_collision(self, point: tuple[int, int]) -> bool:
        x, y = point
        if x < 0 or y < 0 or x >= self.config.grid_size or y >= self.config.grid_size:
            return True
        return point in self.snake[1:]

    def draw_cell(self, x: int, y: int, color: str) -> None:
        size = self.config.cell_size
        self.canvas.create_rectangle(
            x * size,
            y * size,
            (x + 1) * size,
            (y + 1) * size,
            fill=color,
            outline="#1f1f1f",
        )

    def draw(self) -> None:
        self.canvas.delete("all")
        for x in range(self.config.grid_size):
            for y in range(self.config.grid_size):
                self.draw_cell(x, y, "#151515")

        food_x, food_y = self.food
        self.draw_cell(food_x, food_y, "#d64045")

        for index, (x, y) in enumerate(self.snake):
            color = "#5fbf4a" if index == 0 else "#2f8f2f"
            self.draw_cell(x, y, color)

        if self.game_over:
            self.canvas.create_text(
                (self.config.grid_size * self.config.cell_size) // 2,
                (self.config.grid_size * self.config.cell_size) // 2,
                text="GAME OVER",
                fill="white",
                font=("Arial", 28, "bold"),
            )


def play_game() -> None:
    root = tk.Tk()
    root.title("Snake Game")
    root.resizable(False, False)
    SnakeGame(root)
    root.mainloop()


play_game()